In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [2]:
# Load raw dataset
df = pd.read_csv(
    "../artifacts/raw_data/retail_store_inventory.csv"
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (76000, 16)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                76000 non-null  object 
 1   Store ID            76000 non-null  object 
 2   Product ID          76000 non-null  object 
 3   Category            76000 non-null  object 
 4   Region              76000 non-null  object 
 5   Inventory Level     76000 non-null  int64  
 6   Units Sold          76000 non-null  int64  
 7   Units Ordered       76000 non-null  int64  
 8   Price               76000 non-null  float64
 9   Discount            76000 non-null  int64  
 10  Weather Condition   76000 non-null  object 
 11  Promotion           76000 non-null  int64  
 12  Competitor Pricing  76000 non-null  float64
 13  Seasonality         76000 non-null  object 
 14  Epidemic            76000 non-null  int64  
 15  Demand              76000 non-null  int64  
dtypes: f

In [6]:
df.isnull().sum()

Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Price                 0
Discount              0
Weather Condition     0
Promotion             0
Competitor Pricing    0
Seasonality           0
Epidemic              0
Demand                0
dtype: int64

In [7]:
print("\nDuplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [9]:
print("\nDemand statistics:")
df.describe()


Demand statistics:


,Inventory Level,Units Sold,Units Ordered,Price,Discount,Promotion,Competitor Pricing,Epidemic,Demand
count,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000
mean,301.062842,88.827316,89.090645,67.726028,9.087039,0.328947,69.454029,0.200000,104.317158
std,226.510161,43.994525,162.404627,39.377899,7.475781,0.469834,40.943818,0.400003,46.964801
min,0.000000,0.000000,0.000000,4.740000,0.000000,0.000000,4.290000,0.000000,4.000000
25%,136.000000,58.000000,0.000000,31.997500,5.000000,0.000000,32.620000,0.000000,71.000000
50%,227.000000,84.000000,0.000000,64.500000,10.000000,0.000000,65.700000,0.000000,100.000000
75%,408.000000,114.000000,121.000000,95.830000,10.000000,1.000000,97.932500,0.000000,133.000000
max,2267.000000,426.000000,1616.000000,228.030000,25.000000,1.000000,261.220000,1.000000,430.000000


In [11]:
# Convert Date to datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print(df["Date"].dtype)

datetime64[ns]


In [12]:
print("Invalid dates:", df["Date"].isna().sum())

Invalid dates: 0


In [13]:
# Sort Data Chronologically
df = df.sort_values("Date").reset_index(drop=True)

In [16]:
target = "Demand"

In [17]:
print(target)

Demand


In [18]:
leakage_columns = [
    "Units Sold",
    "Units Ordered"
]

In [19]:
df = df.drop(
    columns=leakage_columns,
    errors="ignore"
)

In [20]:
print("Removed potential leakage columns:", leakage_columns)
print(df.columns.tolist())

Removed potential leakage columns: ['Units Sold', 'Units Ordered']
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'Demand']


In [21]:
df.shape

(76000, 14)

In [22]:
# Create Date Features

# Year
df["Year"] = df["Date"].dt.year

# Month number
df["Month"] = df["Date"].dt.month

# Day of month
df["Day"] = df["Date"].dt.day

# Day of week number
df["Day_of_Week"] = df["Date"].dt.dayofweek

# Weekday / Weekend
df["Is_Weekend"] = (df["Day_of_Week"] >= 5).astype(int)

# Week of year
df["Week_of_Year"] = df["Date"].dt.isocalendar().week.astype(int)

In [26]:
# Create Cyclical Time Features

# Cyclical month features
df["Month_Sin"] = np.sin(
    2 * np.pi * df["Month"] / 12
)

df["Month_Cos"] = np.cos(
    2 * np.pi * df["Month"] / 12
)

df["DayOfWeek_Sin"] = np.sin(
    2 * np.pi * df["Day_of_Week"] / 7
)

df["DayOfWeek_Cos"] = np.cos(
    2 * np.pi * df["Day_of_Week"] / 7
)

In [27]:
# Create Competitive Pricing Features

# Price difference
df["Price_Difference"] = (
    df["Price"] - df["Competitor Pricing"]
)


# Price premium percentage
df["Price_Premium_Pct"] = (
    (df["Price"] - df["Competitor Pricing"])
    / df["Competitor Pricing"]
) * 100


# Price Ratio
df["Price_Ratio"] = (
    df["Price"] / df["Competitor Pricing"]
)

In [28]:
# Remove Redundant Competitor Pricing

df = df.drop(
    columns=["Competitor Pricing"],
    errors="ignore"
)

In [29]:
# Pricing Level Features

df["Price_Level"] = pd.qcut(
    df["Price"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

In [30]:
# For discount
df["Discount_Level"] = pd.qcut(
    df["Discount"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

In [31]:
print(df["Price_Level"].value_counts())
print()
print(df["Discount_Level"].value_counts())

Price_Level
Medium    25336
Low       25333
High      25331
Name: count, dtype: int64

Discount_Level
Low       34044
Medium    23298
High      18658
Name: count, dtype: int64


In [32]:
# Promotion + Discount Interaction
df["Promotion_Discount"] = (
    df["Promotion"] * df["Discount"]
)

In [33]:
# Price + Promotion Interaction
df["Price_Promotion"] = (
    df["Price"] * df["Promotion"]
)

In [35]:
# Price + Discount Interaction
df["Price_Discount"] = (
    df["Price"] * df["Discount"]
)

In [36]:
# Category + Seasonality Interaction
df["Category_Seasonality"] = (
    df["Category"].astype(str)
    + "_"
    + df["Seasonality"].astype(str)
)

In [38]:
# Category + Price Interaction
df["Category_Price_Level"] = (
    df["Category"].astype(str)
    + "_"
    + df["Price_Level"].astype(str)
)

In [39]:
# Invenotry Transformation
df["Inventory_Log"] = np.log1p(
    df["Inventory Level"]
)

In [40]:
print("Shape after feature engineering:")
print(df.shape)

Shape after feature engineering:
(76000, 34)


In [41]:
print("\nFinal engineered columns:")
for col in df.columns:
    print(col)


Final engineered columns:
Date
Store ID
Product ID
Category
Region
Inventory Level
Price
Discount
Weather Condition
Promotion
Seasonality
Epidemic
Demand
Year
Month
Day
Day_of_Week
Is_Weekend
Week_of_Year
Month_Sin
Month_Cos
DayOfWeek_Sin
DayOfWeek_Cos
Price_Difference
Price_Premium_Pct
Price_Ratio
Price_Level
Discount_Level
Promotion_Discount
Price_Promotion
Price_Discount
Category_Seasonality
Category_Price_Level
Inventory_Log


In [43]:
# Separate Date for Splitting

model_df = df.copy()

In [44]:
model_df = model_df.sort_values(
    "Date"
).reset_index(drop=True)

In [45]:
# Train-Test Split

split_index = int(
    len(model_df) * 0.80
)

train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (60800, 34)
Testing shape: (15200, 34)


In [46]:
# Train on the past → predict the future.
print(
    "Training period:",
    train_df["Date"].min(),
    "to",
    train_df["Date"].max()
)

print(
    "Testing period:",
    test_df["Date"].min(),
    "to",
    test_df["Date"].max()
)

Training period: 2022-01-01 00:00:00 to 2023-08-31 00:00:00
Testing period: 2023-09-01 00:00:00 to 2024-01-30 00:00:00


In [47]:
# Separate X and y

X_train = train_df.drop(
    columns=["Demand"]
)

y_train = train_df["Demand"]

X_test = test_df.drop(
    columns=["Demand"]
)

y_test = test_df["Demand"]

In [48]:
# Remove raw date
X_train = X_train.drop(
    columns=["Date"]
)

X_test = X_test.drop(
    columns=["Date"]
)

In [49]:
# Identify Numerical Features
numerical_features = X_train.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

Numerical features:
['Inventory Level', 'Price', 'Discount', 'Promotion', 'Epidemic', 'Year', 'Month', 'Day', 'Day_of_Week', 'Is_Weekend', 'Week_of_Year', 'Month_Sin', 'Month_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Price_Difference', 'Price_Premium_Pct', 'Price_Ratio', 'Promotion_Discount', 'Price_Promotion', 'Price_Discount', 'Inventory_Log']


In [50]:
# Identify Categorical Features
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nCategorical features:")
print(categorical_features)


Categorical features:
['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality', 'Price_Level', 'Discount_Level', 'Category_Seasonality', 'Category_Price_Level']


In [51]:
# Numerical Preprocessing
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

# Categorical Preprocessing
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [52]:
# Combine the Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

In [53]:
# Fit only on Training Data
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

In [54]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)

In [55]:
print("Number of final features:", len(feature_names))

Number of final features: 103


In [56]:
# Convert to DataFrames
X_train_final = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_final = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [57]:
# Check final dataset
print("Final X_train shape:")
print(X_train_final.shape)

print("\nFinal X_test shape:")
print(X_test_final.shape)

print("\ny_train shape:")
print(y_train.shape)

print("\ny_test shape:")
print(y_test.shape)

Final X_train shape:
(60800, 103)

Final X_test shape:
(15200, 103)

y_train shape:
(60800,)

y_test shape:
(15200,)


In [62]:
feature_summary = pd.DataFrame({
    "Feature": X_train.columns,
    "Data_Type": [
        X_train[col].dtype
        for col in X_train.columns
    ]
})

print(feature_summary)

                 Feature Data_Type
0               Store ID    object
1             Product ID    object
2               Category    object
3                 Region    object
4        Inventory Level     int64
5                  Price   float64
6               Discount     int64
7      Weather Condition    object
8              Promotion     int64
9            Seasonality    object
10              Epidemic     int64
11                  Year     int32
12                 Month     int32
13                   Day     int32
14           Day_of_Week     int32
15            Is_Weekend     int64
16          Week_of_Year     int64
17             Month_Sin   float64
18             Month_Cos   float64
19         DayOfWeek_Sin   float64
20         DayOfWeek_Cos   float64
21      Price_Difference   float64
22     Price_Premium_Pct   float64
23           Price_Ratio   float64
24           Price_Level  category
25        Discount_Level  category
26    Promotion_Discount     int64
27       Price_Promo

In [64]:
import os

os.makedirs(
    "../artifacts/processed",
    exist_ok=True
)

In [65]:
# Save the Final Feature Sets
X_train_final.to_csv(
    "../artifacts/processed/X_train.csv",
    index=False
)

X_test_final.to_csv(
    "../artifacts/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../artifacts/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../artifacts/processed/y_test.csv",
    index=False
)

In [66]:
import joblib

joblib.dump(
    preprocessor,
    "../artifacts/processed/preprocessor.pkl"
)

['../artifacts/processed/preprocessor.pkl']